# Spark Implementation
# Task 4 & Task 6
Mercy




In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
!apt-get update -qq

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [14]:
!apt-get install openjdk-11-jdk-headless -qq

In [15]:
!pip install -q pyspark

In [16]:
from pyspark import SparkConf, SparkContext

conf = (
    SparkConf()
    .setAppName("Movies")
    .setMaster("local[*]")
    .set("spark.driver.memory", "10g")
)

sc = SparkContext.getOrCreate(conf=conf)
sc.setLogLevel("ERROR")
sc

<SparkContext master=local[*] appName=Movies>

In [17]:
''' You may have to reupload csv file if you restart env!!'''

import pandas as pd
# setup

df = pd.read_csv("/content/sample_data/movies_cleaned.csv", engine='python', on_bad_lines="skip")

In [18]:
print(df.shape) # yall this result keeps changing on me - sometimes less/more

(590202, 166)


# Task 4: Exploding Genres

In [19]:
'''original time: 5.8 seconds'''

# MapReduce
rdd4 = sc.parallelize(df["genres"].dropna())

rdd4 = rdd4.flatMap(lambda x: x.split(", "))
rdd4 = rdd4.filter(lambda g: g is not None and g != "")
rdd4 = rdd4.map(lambda g: (g, 1))
rdd4 = rdd4.reduceByKey(lambda a, b: a + b)

In [27]:
print(rdd4.collect())

[('Comedy', 145339), ('Drama', 248855), ('Romance', 56214), ('Action', 43322), ('Adventure', 23734), ('Science Fiction', 21145), ('Horror', 55473), ('Documentary', 95124), ('Music', 34761), ('History', 16563), ('Crime', 37117), ('Thriller', 51903), ('Animation', 26749), ('Family', 26696), ('Mystery', 22208), ('Fantasy', 23397), ('War', 10351), ('Western', 9241), ('TV Movie', 28249)]


# Task 6: Genre Performance per Production Company

Longer & Shortened Version

In [21]:
# set up... should this have been done in Map Reduce too? idk
df = df[["production_companies_clean", "genres", "id", "vote_average","revenue"]].dropna()
df = df.rename(columns = {"production_companies_clean":"production_company"})
df["production_company"] = df["production_company"].str.strip('[]').str.replace("'", "").str.split(', ')
df = df[df["revenue"]>0]
records = df.to_dict("records")

# Longer (inital) Version
Closer to my python code

In [24]:
'''original time: 20.68'''

# MapReduce
rdd5 = sc.parallelize(records)


# MapReduce: Mapping Stage-------------

# explode production companies
rdd_companies = rdd5.flatMap(lambda x: [
    {
        'production_company': company,
        'genres': x['genres'],
        'id': x['id'],
        'vote_average': x['vote_average'],
        'revenue': x['revenue']
    }
    for company in x['production_company']
])
# explode genres
rdd_companyGenre = rdd_companies.flatMap(lambda x: [
    {
        'production_company': x['production_company'],
        'genre': genre.strip(),
        'id': x['id'],
        'vote_average': x['vote_average'],
        'revenue': x['revenue']
    }
    for genre in x['genres'].split(', ')
])
# key value pairs
rdd_companyGenre = rdd_companyGenre.map(lambda x: (
    (x['production_company'], x['genre']),
    (1, x['vote_average'], x['revenue'])  # count, rating, revenue
))



# MapReduce: Shuffle + Reduce Stage---------------

rdd_companyGenre = rdd_companyGenre.reduceByKey(lambda a, b: (
    a[0] + b[0],           # sum counts
    a[1] + b[1],           # sum ratings (avg calculated later)
    a[2] + b[2]            # sum revenues
))
# ordering the output
rdd_companyGenre = rdd_companyGenre.map(lambda x: {
    'production_company': x[0][0],
    'genres': x[0][1],
    'movie_count': x[1][0],
    'avg_rating': round(x[1][1] / x[1][0], 2),  # ratings avg
    'total_rev': x[1][2]
})
# sorting by production companies and movie count
rdd_companyGenre = rdd_companyGenre.sortBy(
    lambda x: (x['production_company'], -x['movie_count'])
)
# MapReduce: Output
rdd_companyGenre.take(5)


[{'production_company': '"A.N.P.A. (Agence Nationale de Promotion de lAudiovisuel)"',
  'genres': 'Music',
  'movie_count': 1,
  'avg_rating': 6.0,
  'total_rev': 3314254},
 {'production_company': '"A.N.P.A. (Agence Nationale de Promotion de lAudiovisuel)"',
  'genres': 'Drama',
  'movie_count': 1,
  'avg_rating': 6.0,
  'total_rev': 3314254},
 {'production_company': '"Agence Nationale pour la Cohésion Sociale et lEgalité des Chances (ACSE)"',
  'genres': 'Drama',
  'movie_count': 1,
  'avg_rating': 6.1,
  'total_rev': 502392},
 {'production_company': '"Alin"',
  'genres': 'Drama',
  'movie_count': 1,
  'avg_rating': 6.8,
  'total_rev': 1370693},
 {'production_company': '"Alin"',
  'genres': 'Music',
  'movie_count': 1,
  'avg_rating': 6.8,
  'total_rev': 1370693}]

# Shorter (improved?) version

In [23]:
'''original time: 20.68'''

#MapReduce
rdd = sc.parallelize(records)

# MapReduce: Mapping Stage-------------

# explode
rdd = rdd.flatMap(lambda x: [
    {
        "production_company": company,
        "genres": x["genres"],
        "vote_average": x["vote_average"],
        "revenue": x["revenue"]
    }
    for company in x["production_company"]
])
rdd = rdd.flatMap(lambda x: [
    (
        (x["production_company"], genre.strip()),
        (1, x["vote_average"], x["revenue"])
    )
    for genre in x["genres"].split(", ")
])

# MapReduce: Shuffle + Reduce Stage---------------
rdd = rdd.reduceByKey(lambda a, b: (
    a[0] + b[0],      # movie count
    a[1] + b[1],      # sum ratings
    a[2] + b[2]       # sum revenue
))

# Ordering and sorting
rdd = rdd.map(lambda x: {
    "production_company": x[0][0],
    "genre": x[0][1],
    "movie_count": x[1][0],
    "avg_rating": round(x[1][1] / x[1][0], 2),
    "total_rev": x[1][2]
})

rdd = rdd.sortBy(lambda x: (x["production_company"], -x["movie_count"]))

# MapReduce: Output
rdd.take(5)

[{'production_company': '"A.N.P.A. (Agence Nationale de Promotion de lAudiovisuel)"',
  'genre': 'Music',
  'movie_count': 1,
  'avg_rating': 6.0,
  'total_rev': 3314254},
 {'production_company': '"A.N.P.A. (Agence Nationale de Promotion de lAudiovisuel)"',
  'genre': 'Drama',
  'movie_count': 1,
  'avg_rating': 6.0,
  'total_rev': 3314254},
 {'production_company': '"Agence Nationale pour la Cohésion Sociale et lEgalité des Chances (ACSE)"',
  'genre': 'Drama',
  'movie_count': 1,
  'avg_rating': 6.1,
  'total_rev': 502392},
 {'production_company': '"Alin"',
  'genre': 'Drama',
  'movie_count': 1,
  'avg_rating': 6.8,
  'total_rev': 1370693},
 {'production_company': '"Alin"',
  'genre': 'Music',
  'movie_count': 1,
  'avg_rating': 6.8,
  'total_rev': 1370693}]